# Iteam Seach Exploration and Tests Notebook

In this notebook I'm basically just exploring the data and testing functions.

## Data Exploration

In [1]:
import os
import duckdb

DATA_DIR = os.environ.get("DATA_DIR", "../data")  # matches the dbt convention

conn = duckdb.connect()  # in-memory, just for exploring

# Register CSVs as views so you can query with SQL as if they were tables.
# Swap these paths for data/silver/... once you've run dbt if you want the typed versions.
conn.sql(f"""
    create view dim_item as
    select * from read_csv_auto('{DATA_DIR}/gold/dim_item.csv')
""")
conn.sql(f"""
    create view search_queries as
    select * from read_csv_auto('{DATA_DIR}/search_queries.csv')
""")

In [5]:
conn.sql("SELECT * FROM dim_item LIMIT 5")

┌──────────┬───────────────────────────────────┬──────────────┬─────────────────┬─────────────────┬───────────────────┐
│ item_id  │            description            │   category   │ unit_of_measure │ manufacturer_id │ manufacturer_name │
│ varchar  │              varchar              │   varchar    │     varchar     │     varchar     │      varchar      │
├──────────┼───────────────────────────────────┼──────────────┼─────────────────┼─────────────────┼───────────────────┤
│ ITM-0101 │ THHN Wire 14 AWG Black 500ft Reel │ Wire & Cable │ RL              │ MFG-01          │ NorthWire Co.     │
│ ITM-0102 │ THHN Wire 14 AWG White 500ft Reel │ Wire & Cable │ RL              │ MFG-01          │ NorthWire Co.     │
│ ITM-0103 │ THHN Wire 14 AWG Red 500ft Reel   │ Wire & Cable │ RL              │ MFG-01          │ NorthWire Co.     │
│ ITM-0104 │ THHN Wire 14 AWG Green 500ft Reel │ Wire & Cable │ RL              │ MFG-01          │ NorthWire Co.     │
│ ITM-0105 │ THHN Wire 14 AWG Blue 500ft

In [9]:
conn.sql("SELECT COUNT(*) FROM dim_item")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          358 │
└──────────────┘

In [6]:
conn.sql("SELECT * FROM search_queries LIMIT 5")

┌──────────┬──────────────┬───────────────────────────────┬───────────────────┬─────────────────────────────────────────────────────────────────────────┐
│ query_id │  query_type  │             query             │ expected_item_ids │                                  notes                                  │
│ varchar  │   varchar    │            varchar            │      varchar      │                                 varchar                                 │
├──────────┼──────────────┼───────────────────────────────┼───────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Q01      │ abbreviation │ j-hook cable support 2"       │ ITM-0920          │ abbreviation / unit / synonym variant of an item with no near-duplicate │
│ Q02      │ no_match     │ copper pipe 3/4 type l        │ NULL              │ nothing in the catalog should score highly                              │
│ Q03      │ misspelled   │ occupancy sensor swatch ivory │ ITM-0440        

In [12]:
conn.sql("SELECT COUNT(*) FROM search_queries")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│           40 │
└──────────────┘

In [11]:
conn.sql("SELECT query_type, COUNT(*) n_queries FROM search_queries GROUP BY 1")

┌──────────────┬───────────┐
│  query_type  │ n_queries │
│   varchar    │   int64   │
├──────────────┼───────────┤
│ misspelled   │         8 │
│ partial      │         8 │
│ exact        │         8 │
│ abbreviation │         8 │
│ no_match     │         4 │
│ word_order   │         4 │
└──────────────┴───────────┘

In [18]:
conn.sql("SELECT query, expected_item_ids, notes FROM search_queries WHERE query_type = 'misspelled'")

┌────────────────────────────────────┬───────────────────┬──────────────────────────────┐
│               query                │ expected_item_ids │            notes             │
│              varchar               │      varchar      │           varchar            │
├────────────────────────────────────┼───────────────────┼──────────────────────────────┤
│ occupancy sensor swatch ivory      │ ITM-0440          │ one edit in token 'Switch'   │
│ 4-gang dovice box metal            │ ITM-0312          │ one edit in token 'Device'   │
│ pvc couling 2in                    │ ITM-0247          │ one edit in token 'Coupling' │
│ pvc condut elbow 1/2in 90deg       │ ITM-0234          │ one edit in token 'Conduit'  │
│ mc cable 10/2 alumnium armor 250ft │ ITM-0144          │ one edit in token 'Aluminum' │
│ 30a stngle pole circuit breaker    │ ITM-0503|ITM-0535 │ one edit in token 'Single'   │
│ 4-way switch 15a aomond            │ ITM-0411          │ one edit in token 'Almond'   │
│ condnit 

In [17]:
conn.sql("SELECT * FROM dim_item WHERE item_id = 'ITM-0707'")

┌──────────┬───────────────────────────┬──────────┬─────────────────┬─────────────────┬───────────────────────┐
│ item_id  │        description        │ category │ unit_of_measure │ manufacturer_id │   manufacturer_name   │
│ varchar  │          varchar          │ varchar  │     varchar     │     varchar     │        varchar        │
├──────────┼───────────────────────────┼──────────┼─────────────────┼─────────────────┼───────────────────────┤
│ ITM-0707 │ LED Troffer 2x4 5000K 40W │ Lighting │ EA              │ MFG-09          │ Luminex Fixtures Inc. │
└──────────┴───────────────────────────┴──────────┴─────────────────┴─────────────────┴───────────────────────┘

In [22]:
conn.sql(
"""
SELECT CONCAT(
'item_id: ',
item_id,
' | ',
'description: ',
description,
' | ',
'category: ',
category,
' | ',
'manufacturer_id: ',
manufacturer_id,
' | ',
'manufacturer_name: ',
manufacturer_name) item_info
FROM dim_item
WHERE item_id = 'ITM-0707'
"""
)

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                                      item_info                                                                       │
│                                                                       varchar                                                                        │
├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ item_id: ITM-0707 | description: LED Troffer 2x4 5000K 40W | category: Lighting | manufacturer_id: MFG-09 | manufacturer_name: Luminex Fixtures Inc. │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘

## Latency tests on input queries

In [35]:
%%timeit -r 5 -n 100

unique_categories = conn.sql(
"""
SELECT DISTINCT category
FROM dim_item
WHERE category IS NOT NULL
"""
).df()["category"]

unique_manufacturers = conn.sql(
"""
SELECT DISTINCT manufacturer_name
FROM dim_item
WHERE manufacturer_name IS NOT NULL
"""
).df()["manufacturer_name"]

44 ms ± 799 μs per loop (mean ± std. dev. of 5 runs, 100 loops each)


In [42]:
%%timeit -r 5 -n 100

unique_categories = conn.sql(
"""
SELECT DISTINCT category
FROM dim_item
WHERE category IS NOT NULL
"""
).fetchnumpy()["category"]

unique_manufacturers = conn.sql(
"""
SELECT DISTINCT manufacturer_name
FROM dim_item
WHERE manufacturer_name IS NOT NULL
"""
).fetchnumpy()["manufacturer_name"]

44.8 ms ± 1.55 ms per loop (mean ± std. dev. of 5 runs, 100 loops each)


In [39]:
%%timeit -r 5 -n 100

df = conn.sql(
"""
SELECT DISTINCT 'category' column_source, 
category col_value
FROM dim_item
WHERE category IS NOT NULL
UNION ALL
SELECT DISTINCT 'manufacturer_name',
manufacturer_name col_value
FROM dim_item
WHERE manufacturer_name IS NOT NULL;
"""
).df()

unique_categories = df[df["column_source"] == "category"]["col_value"]
unique_manufacturers = df[df["column_source"] == "manufacturer_name"]["col_value"]


44.3 ms ± 1.18 ms per loop (mean ± std. dev. of 5 runs, 100 loops each)


## Router Test 

In [56]:
unique_categories = conn.sql(
"""
SELECT DISTINCT LOWER(category) category
FROM dim_item
WHERE category IS NOT NULL
"""
).df()["category"]

unique_manufacturers = conn.sql(
"""
SELECT DISTINCT LOWER(manufacturer_name) manufacturer_name
FROM dim_item
WHERE manufacturer_name IS NOT NULL
"""
).df()["manufacturer_name"]

In [82]:
import re
from rapidfuzz import fuzz, process

# built once at index time, cached alongside your embeddings
SKU_PATTERN = re.compile(r"^itm-\d{4}$")   # confirm the real format from dim_item.item_id
MANUFACTURER_PATTERN = re.compile(r"^MFG-\d{2}$")
CATEGORIES = sorted(unique_categories)
MANUFACTURERS = sorted(unique_manufacturers)

def classify_query(query: str, category_threshold=90, manufacturer_threshold=90):
    q = query.strip().lower()

    # 1. structural check — no ambiguity possible, so it goes first and short-circuits
    if SKU_PATTERN.fullmatch(q):
        return "sku", q.upper(), 100

    if MANUFACTURER_PATTERN.fullmatch(q):
        return "manufacturer_id", q.upper(), 100

    # 2. closed vocab — whole-string fuzz.ratio, not partial_ratio, so a description
    #    query that merely *contains* a category word doesn't misfire
    cat_match, cat_score, _ = process.extractOne(q, CATEGORIES, scorer=fuzz.ratio)
    if cat_score >= category_threshold:
        return "category", cat_match, cat_score

    mfg_match, mfg_score, _ = process.extractOne(q, MANUFACTURERS, scorer=fuzz.ratio)
    if mfg_score >= manufacturer_threshold:
        return "manufacturer", mfg_match, mfg_score

    # 3. catch-all — everything else, including malformed SKU-like queries
    return "description", q, (cat_score, mfg_score)

In [47]:
conn.sql(
"""
    SELECT query,
    expected_item_ids,
    notes
    FROM search_queries
    WHERE query_type = 'misspelled'
"""
)

┌────────────────────────────────────┬───────────────────┬──────────────────────────────┐
│               query                │ expected_item_ids │            notes             │
│              varchar               │      varchar      │           varchar            │
├────────────────────────────────────┼───────────────────┼──────────────────────────────┤
│ occupancy sensor swatch ivory      │ ITM-0440          │ one edit in token 'Switch'   │
│ 4-gang dovice box metal            │ ITM-0312          │ one edit in token 'Device'   │
│ pvc couling 2in                    │ ITM-0247          │ one edit in token 'Coupling' │
│ pvc condut elbow 1/2in 90deg       │ ITM-0234          │ one edit in token 'Conduit'  │
│ mc cable 10/2 alumnium armor 250ft │ ITM-0144          │ one edit in token 'Aluminum' │
│ 30a stngle pole circuit breaker    │ ITM-0503|ITM-0535 │ one edit in token 'Single'   │
│ 4-way switch 15a aomond            │ ITM-0411          │ one edit in token 'Almond'   │
│ condnit 

In [51]:
conn.sql("SELECT * FROM dim_item WHERE item_id = 'ITM-0440'")

┌──────────┬───────────────────────────────┬────────────────┬─────────────────┬─────────────────┬────────────────────┐
│ item_id  │          description          │    category    │ unit_of_measure │ manufacturer_id │ manufacturer_name  │
│ varchar  │            varchar            │    varchar     │     varchar     │     varchar     │      varchar       │
├──────────┼───────────────────────────────┼────────────────┼─────────────────┼─────────────────┼────────────────────┤
│ ITM-0440 │ Occupancy Sensor Switch Ivory │ Wiring Devices │ EA              │ MFG-06          │ Circuitide Devices │
└──────────┴───────────────────────────────┴────────────────┴─────────────────┴─────────────────┴────────────────────┘

In [83]:
# Exact SKU match
classify_query("ITM-0440",
category_threshold=90,
manufacturer_threshold=90
)

('sku', 'ITM-0440', 100)

In [84]:
# Incorrect SKU (wrong length)
classify_query("ITM-044011",
category_threshold=90,
manufacturer_threshold=90
)

('description', 'itm-044011', (22.22222222222222, 14.28571428571429))

In [85]:
# Description
classify_query("pvc couling 2in",
category_threshold=90,
manufacturer_threshold=90
)

('description', 'pvc couling 2in', (43.47826086956522, 38.888888888888886))

In [86]:
# Category exact match
classify_query("Wiring Devices",
category_threshold=90,
manufacturer_threshold=90
)

('category', 'wiring devices', 100.0)

In [87]:
# Category misspelling
classify_query("wiing devices",
category_threshold=90,
manufacturer_threshold=90
)

('category', 'wiring devices', 96.2962962962963)

In [88]:
# Category misspelling worse
classify_query("wiing evices",
category_threshold=90,
manufacturer_threshold=90
)

('category', 'wiring devices', 92.3076923076923)

In [89]:
# Category abbreviation
classify_query("boxes",
category_threshold=90,
manufacturer_threshold=90
)

('description', 'boxes', (43.47826086956522, 35.71428571428571))

In [90]:
# Category abbreviation
classify_query("circuit",
category_threshold=90,
manufacturer_threshold=90
)

('description', 'circuit', (56.00000000000001, 56.00000000000001))

In [91]:
# Category abbreviation
classify_query("boxes enclosures",
category_threshold=90,
manufacturer_threshold=90
)

('category', 'boxes & enclosures', 94.11764705882352)

In [92]:
# Category abbreviation
classify_query("boxes and enclosures",
category_threshold=90,
manufacturer_threshold=90
)

('description', 'boxes and enclosures', (89.47368421052632, 46.51162790697675))

In [ ]:
# Category abbreviation
classify_query("lighting",
category_threshold=87,
manufacturer_threshold=87
)

('category', 'lighting', 100.0)

In [64]:
CATEGORIES

['boxes & enclosures',
 'circuit protection',
 'conduit & fittings',
 'connectors & lugs',
 'fasteners & supports',
 'grounding',
 'lighting',
 'tape & sealants',
 'wire & cable',
 'wiring devices']

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent  # notebooks/ -> repo root
sys.path.insert(0, str(ROOT))

from src.item_search.router import classify_query as classify_query_router
from src.item_search.index import build_index

In [103]:
idx = build_index(conn)
classify_query_router("boxes enclosures", idx)

('category', 'boxes & enclosures', 94.11764705882352)

In [106]:
classify_query_router("ITM-0449", idx)

('sku_not_found', 'ITM-0449', 0)

In [109]:
classify_query_router("groding", idx)

('description', 'groding', (87.5, 45.16129032258065))

In [114]:
TOKEN_RE = re.compile(r"[a-z0-9]+")
TOKEN_RE.findall("HelloWorld,how are|you?Are you ready? 11 11111".lower())

['helloworld', 'how', 'are', 'you', 'are', 'you', 'ready', '11', '11111']

In [115]:
conn.sql("SELECT COUNT(*) FROM dim_item")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          358 │
└──────────────┘

## Popularity queries

In [117]:
conn.sql(f"""
    select * from read_csv_auto('{DATA_DIR}/gold/item_purchase_history.csv') LIMIT 5
""")

┌──────────┬────────────────────┬────────────────────────┬──────────────────────┬───────────┬───────────┬───────────┬───────────┐
│ item_id  │ purchase_frequency │ earliest_purchase_date │ latest_purchase_date │ price_p25 │ price_p50 │ price_p75 │ price_p90 │
│ varchar  │       int64        │          date          │         date         │  double   │  double   │  double   │  double   │
├──────────┼────────────────────┼────────────────────────┼──────────────────────┼───────────┼───────────┼───────────┼───────────┤
│ ITM-0104 │                 57 │ 2024-01-03             │ 2026-09-01           │     64.99 │      67.3 │     71.03 │     74.06 │
│ ITM-0201 │                177 │ 2024-01-02             │ 2026-09-01           │      4.07 │       4.2 │      4.34 │      4.51 │
│ ITM-0208 │                 40 │ 2024-01-10             │ 2026-08-01           │      4.87 │       5.1 │      5.27 │      5.55 │
│ ITM-0211 │                 27 │ 2024-01-29             │ 2026-07-18           │     12.3

In [3]:
conn.sql(f"""
    select * from read_csv_auto('{DATA_DIR}/silver/itemized_purchase_orders.csv') LIMIT 5
""")

┌───────────┬─────────────┬──────────┬──────────────────┬─────────────────┐
│ po_number │ customer_id │ item_id  │ quantity_ordered │ unit_price_paid │
│  varchar  │   varchar   │ varchar  │      int64       │     double      │
├───────────┼─────────────┼──────────┼──────────────────┼─────────────────┤
│ PO-100000 │ CUST-A      │ ITM-0431 │               10 │            0.72 │
│ PO-100001 │ CUST-B      │ ITM-0220 │               10 │            2.66 │
│ PO-100001 │ CUST-B      │ ITM-0426 │                4 │           26.05 │
│ PO-100001 │ CUST-B      │ ITM-0612 │                2 │            4.71 │
│ PO-100001 │ CUST-B      │ ITM-1015 │               12 │             4.8 │
└───────────┴─────────────┴──────────┴──────────────────┴─────────────────┘

In [4]:
conn.sql(f"""
    SELECT IPO.po_number,
    PO.order_date,
    IPO.item_id,
    IPO.quantity_ordered
    FROM read_csv_auto('{DATA_DIR}/silver/itemized_purchase_orders.csv') IPO
    LEFT JOIN read_csv_auto('{DATA_DIR}/silver/purchase_orders.csv') PO
        ON IPO.po_number = PO.po_number
    ORDER BY 2 ASC
    LIMIT 10
""")

┌───────────┬────────────┬──────────┬──────────────────┐
│ po_number │ order_date │ item_id  │ quantity_ordered │
│  varchar  │    date    │ varchar  │      int64       │
├───────────┼────────────┼──────────┼──────────────────┤
│ PO-101235 │ 2024-01-01 │ ITM-0434 │                6 │
│ PO-102140 │ 2024-01-02 │ ITM-0228 │                6 │
│ PO-102428 │ 2024-01-02 │ ITM-0219 │                4 │
│ PO-102140 │ 2024-01-02 │ ITM-0434 │               10 │
│ PO-102428 │ 2024-01-02 │ ITM-0903 │               12 │
│ PO-102428 │ 2024-01-02 │ ITM-0201 │                6 │
│ PO-102140 │ 2024-01-02 │ ITM-0929 │                5 │
│ PO-103099 │ 2024-01-02 │ ITM-0220 │                4 │
│ PO-103099 │ 2024-01-02 │ ITM-0228 │                4 │
│ PO-103099 │ 2024-01-02 │ ITM-0237 │               10 │
└───────────┴────────────┴──────────┴──────────────────┘
  10 rows                                    4 columns

## Manual Tests on search engine

In [3]:
from src.item_search.search_pipeline import search_item

In [8]:
conn.sql("SELECT * FROM dim_item WHERE item_id = 'ITM-0228'")

┌──────────┬────────────────────────────┬────────────────────┬─────────────────┬─────────────────┬──────────────────────┬────────────────────────────────────────────────────────────────────┐
│ item_id  │        description         │      category      │ unit_of_measure │ manufacturer_id │  manufacturer_name   │                            search_text                             │
│ varchar  │          varchar           │      varchar       │     varchar     │     varchar     │       varchar        │                              varchar                               │
├──────────┼────────────────────────────┼────────────────────┼─────────────────┼─────────────────┼──────────────────────┼────────────────────────────────────────────────────────────────────┤
│ ITM-0228 │ EMT Coupling 2in Set Screw │ Conduit & fittings │ EA              │ MFG-04          │ Ferrotube industries │ EMT Coupling 2in Set Screw Conduit & fittings Ferrotube industries │
└──────────┴────────────────────────────┴────

In [4]:
query = "ITM-0228"
search_item(query)

[{'query': 'ITM-0228',
  'intent': 'sku',
  'matched_value': 'ITM-0228',
  'item_id': 'ITM-0228',
  'item_score': 1.0,
  'route_score': 100}]

In [6]:
query = "ITM-02280"
search_item(query)

[]

In [7]:
query = "EMT Coupling 2in Set Screw"
search_item(query)

[{'query': 'EMT Coupling 2in Set Screw',
  'intent': 'description',
  'matched_value': 'emt coupling 2in set screw',
  'item_id': 'ITM-0228',
  'item_score': 0.8480450510978699,
  'route_score': None},
 {'query': 'EMT Coupling 2in Set Screw',
  'intent': 'description',
  'matched_value': 'emt coupling 2in set screw',
  'item_id': 'ITM-0223',
  'item_score': 0.8254651896566751,
  'route_score': None},
 {'query': 'EMT Coupling 2in Set Screw',
  'intent': 'description',
  'matched_value': 'emt coupling 2in set screw',
  'item_id': 'ITM-0227',
  'item_score': 0.8032002417061976,
  'route_score': None},
 {'query': 'EMT Coupling 2in Set Screw',
  'intent': 'description',
  'matched_value': 'emt coupling 2in set screw',
  'item_id': 'ITM-0225',
  'item_score': 0.7579159850087364,
  'route_score': None},
 {'query': 'EMT Coupling 2in Set Screw',
  'intent': 'description',
  'matched_value': 'emt coupling 2in set screw',
  'item_id': 'ITM-0224',
  'item_score': 0.7422657745162842,
  'route_score

In [10]:
conn.sql("SELECT * FROM dim_item WHERE item_id = 'ITM-0223'")

┌──────────┬──────────────────────────────┬────────────────────┬─────────────────┬─────────────────┬──────────────────────┬──────────────────────────────────────────────────────────────────────┐
│ item_id  │         description          │      category      │ unit_of_measure │ manufacturer_id │  manufacturer_name   │                             search_text                              │
│ varchar  │           varchar            │      varchar       │     varchar     │     varchar     │       varchar        │                               varchar                                │
├──────────┼──────────────────────────────┼────────────────────┼─────────────────┼─────────────────┼──────────────────────┼──────────────────────────────────────────────────────────────────────┤
│ ITM-0223 │ EMT Coupling 1/2in Set Screw │ Conduit & fittings │ EA              │ MFG-04          │ Ferrotube industries │ EMT Coupling 1/2in Set Screw Conduit & fittings Ferrotube industries │
└──────────┴─────────────

In [8]:
query = "Ferrotube industries"
search_item(query)

[{'query': 'Ferrotube industries',
  'intent': 'manufacturer',
  'matched_value': 'ferrotube industries',
  'item_id': 'ITM-0230',
  'item_score': None,
  'route_score': 100.0},
 {'query': 'Ferrotube industries',
  'intent': 'manufacturer',
  'matched_value': 'ferrotube industries',
  'item_id': 'ITM-0228',
  'item_score': None,
  'route_score': 100.0},
 {'query': 'Ferrotube industries',
  'intent': 'manufacturer',
  'matched_value': 'ferrotube industries',
  'item_id': 'ITM-0243',
  'item_score': None,
  'route_score': 100.0},
 {'query': 'Ferrotube industries',
  'intent': 'manufacturer',
  'matched_value': 'ferrotube industries',
  'item_id': 'ITM-0237',
  'item_score': None,
  'route_score': 100.0},
 {'query': 'Ferrotube industries',
  'intent': 'manufacturer',
  'matched_value': 'ferrotube industries',
  'item_id': 'ITM-0201',
  'item_score': None,
  'route_score': 100.0},
 {'query': 'Ferrotube industries',
  'intent': 'manufacturer',
  'matched_value': 'ferrotube industries',
  'i

In [9]:
query = "MFG-04"
search_item(query)

[{'query': 'MFG-04',
  'intent': 'manufacturer_id',
  'matched_value': 'MFG-04',
  'item_id': 'ITM-0230',
  'item_score': None,
  'route_score': 100},
 {'query': 'MFG-04',
  'intent': 'manufacturer_id',
  'matched_value': 'MFG-04',
  'item_id': 'ITM-0228',
  'item_score': None,
  'route_score': 100},
 {'query': 'MFG-04',
  'intent': 'manufacturer_id',
  'matched_value': 'MFG-04',
  'item_id': 'ITM-0243',
  'item_score': None,
  'route_score': 100},
 {'query': 'MFG-04',
  'intent': 'manufacturer_id',
  'matched_value': 'MFG-04',
  'item_id': 'ITM-0237',
  'item_score': None,
  'route_score': 100},
 {'query': 'MFG-04',
  'intent': 'manufacturer_id',
  'matched_value': 'MFG-04',
  'item_id': 'ITM-0201',
  'item_score': None,
  'route_score': 100},
 {'query': 'MFG-04',
  'intent': 'manufacturer_id',
  'matched_value': 'MFG-04',
  'item_id': 'ITM-0225',
  'item_score': None,
  'route_score': 100},
 {'query': 'MFG-04',
  'intent': 'manufacturer_id',
  'matched_value': 'MFG-04',
  'item_id': 

In [10]:
query = "Conduit & fittings"
search_item(query)

[{'query': 'Conduit & fittings',
  'intent': 'category',
  'matched_value': 'conduit & fittings',
  'item_id': 'ITM-0230',
  'item_score': None,
  'route_score': 100.0},
 {'query': 'Conduit & fittings',
  'intent': 'category',
  'matched_value': 'conduit & fittings',
  'item_id': 'ITM-0228',
  'item_score': None,
  'route_score': 100.0},
 {'query': 'Conduit & fittings',
  'intent': 'category',
  'matched_value': 'conduit & fittings',
  'item_id': 'ITM-0220',
  'item_score': None,
  'route_score': 100.0},
 {'query': 'Conduit & fittings',
  'intent': 'category',
  'matched_value': 'conduit & fittings',
  'item_id': 'ITM-0243',
  'item_score': None,
  'route_score': 100.0},
 {'query': 'Conduit & fittings',
  'intent': 'category',
  'matched_value': 'conduit & fittings',
  'item_id': 'ITM-0237',
  'item_score': None,
  'route_score': 100.0},
 {'query': 'Conduit & fittings',
  'intent': 'category',
  'matched_value': 'conduit & fittings',
  'item_id': 'ITM-0235',
  'item_score': None,
  'ro

In [11]:
query = "Screw"
search_item(query)

[{'query': 'Screw',
  'intent': 'description',
  'matched_value': 'screw',
  'item_id': 'ITM-0628',
  'item_score': 0.6513979434967041,
  'route_score': None},
 {'query': 'Screw',
  'intent': 'description',
  'matched_value': 'screw',
  'item_id': 'ITM-0934',
  'item_score': 0.6252809156971857,
  'route_score': None},
 {'query': 'Screw',
  'intent': 'description',
  'matched_value': 'screw',
  'item_id': 'ITM-0941',
  'item_score': 0.6244624748675356,
  'route_score': None},
 {'query': 'Screw',
  'intent': 'description',
  'matched_value': 'screw',
  'item_id': 'ITM-0933',
  'item_score': 0.6054498448817263,
  'route_score': None},
 {'query': 'Screw',
  'intent': 'description',
  'matched_value': 'screw',
  'item_id': 'ITM-0936',
  'item_score': 0.595113394517873,
  'route_score': None}]